# 00 · Blueprint v2: per-event encoders under a causal history decoder

**Blueprint v2 tour, the pragmatiq series' overview. The [Blueprint v1 notebooks](../blueprint_v1/README.md) (02 to 05) built the original Blueprint (v1): every field of every transaction is one token in a long flat stream, read by a Llama decoder. This notebook tours its successor, Blueprint v2, building block by building block, on a pretrained checkpoint.**

Blueprint v1's notebook 05 ended with a puzzle. Hand-built aggregates of a customer's history helped a fraud model, but the history embedding barely did. One suspect: the last-token state after 1,537 tokens, where the scored transaction is 12 tokens among 127 earlier transactions. Blueprint v2 changes the shape of the problem. A small **bidirectional encoder** compresses each event, and the customer's profile, into **one vector**. A **causal decoder** then reads one vector per event, in time order, and predicts the next event. A history of 128 events is 129 positions, not 1,537 tokens.

The design borrows PRAGMA's per-event encoders (Revolut's encoder-only transaction model) and keeps v1's causal next-event pretraining, so the embedding at a cutoff still depends only on the past.

In this notebook we:
1. **Compare** v1 and v2 building block by building block
2. **Meet the data**: a synthetic multi-source bank book with customer profiles and five downstream tasks
3. **Tokenize** one event as key-value tokens, and see how time enters without being a token
4. **Open the network**: profile encoder, event encoder, causal decoder, and where the parameters are
5. **Check what pretraining taught** the model, against the same network untrained
6. **Fine-tune** the pretrained model with LoRA on each task, and score it against XGBoost on raw features

About 10 minutes of compute on one GPU and about 30 GB of host memory: preparing a 4,000-user sample, then a short LoRA fine-tune per task. Nothing is pretrained here. `fetch_lab_assets` finds the book and the checkpoint ([README](README.md)). Every section has a notebook in this series that goes further.

> **Smoke mode.** CI runs this notebook with `TFM_NOTEBOOK_SMOKE=1`: a
> generated 120-user book, an untrained model, and CPU only. Every cell runs,
> but the numbers and plots are meaningless. The first code cell prints which
> mode you are in.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from tfm.runtime.lab import fetch_lab_assets, lab_tasks, load_lab_model, prepare_lab_data
from tfm.training.reproducibility import seed_everything

assets = fetch_lab_assets("00")  # 4,000 users sampled from the book's 100,000
config = assets.config
seed_everything(config.seed)
print(assets.describe())

started = time.perf_counter()
data = prepare_lab_data(assets)  # load, validate, split by time, tokenize, build sequences
tokenizer = data.tokenizer
model = load_lab_model(assets, data)                             # the pretrained checkpoint
random_model = load_lab_model(assets, data, pretrained=False)   # same network, random weights
profiles = {p.entity_id: p for p in data.profiles}
print(f"{len(data.events):,} events from {len(data.profiles):,} users prepared in {time.perf_counter() - started:.0f} s")

## 1. From v1 to v2, building block by building block

The Blueprint v1 notebooks (02 to 05) walked v1's five building blocks. Blueprint v2 changes all but the readout:

| Building block | Blueprint v1 (v1 notebooks 02–05) | Blueprint v2 (this notebook) |
|---|---|---|
| Data | TabFormer: one card-transaction table, no profiles | pragmatiq book: profiles plus card, bank, app, marketing, and trading events |
| Event representation | 11 fixed field tokens per transaction, plus `<sep>` | key-value tokens, any fields per event; identifiers hashed; time as a rotary position, not a token |
| Sequence architecture | one causal Llama over the flat token stream (1,537 tokens for 128 transactions) | bidirectional profile and event encoders, one vector each, under a causal decoder (129 positions) |
| Objective | next token | next event's field values, plus PRAGMA's masked tokens inside each event |
| Readout and adaptation | last token; XGBoost on raw + embedding | last slot; **LoRA fine-tuning** of the whole network, one copy per task |

```text
v1:  <bos> AMT_3 MERCH_972 … STATE_NJ <sep> AMT_1 … <sep> … <eos>  ──► causal Llama (one position per token)

v2:  profile tokens (as of the cutoff) ──► profile encoder ──► z_profile ┐
     event j key/value tokens         ──► event encoder   ──► z_event[j] ┴─► causal decoder (one slot per event)
                                                                              + calendar, TimeRoPE(Δt) ──► h
```

## 2. The data: a multi-source book

v2 needs a profile per customer and events from several sources, which TabFormer doesn't have. The lab uses a **pragmatiq** book: 100,000 synthetic customers over 31 months, from an agent-based simulator (pragmatiq 1.1.0, Apache-2.0). No real customer is in it. `prepare_lab_data` ran the whole data block above, on a 4,000-customer sample: validate every event, split by time into train, validation, and test periods, tokenize, and build one sequence per customer at each cutoff.

A customer is a profile and one mixed event stream:

In [ ]:
user = data.test_sequences[0].entity_id
print("profile:", profiles[user].attributes)

examples = {}
for event in data.events[:200_000]:
    examples.setdefault(event.categorical.get("source", "unknown"), event)
for source, event in examples.items():
    fields = {**event.categorical, **{k: round(v, 2) for k, v in event.numeric.items()}}
    fields.pop("source", None)
    print(f"{source:<12} {event.event_timestamp:%Y-%m-%d %H:%M}  {fields}")

A card payment, an app session, and a marketing message carry different fields. v1's fixed 11-field layout can't hold them, which is why v2's tokenizer has no fixed schema.

**Five tasks on the same customers.** The book carries forecast labels, so one pretrained model can be tested on several tasks:

| Task | One row is… | Label |
|---|---|---|
| `fraud_user` | a customer's history up to the end of the label window | 1 if an identity thief's payments were spliced into it |
| `fraud_txn` | one labelled card transaction, with the history up to and including it (as in Blueprint v1's notebook 05) | 1 if the transaction was copied in from another customer |
| `churn_6m` | a customer's history at 2025-01-01 | 1 if the customer goes quiet in the next six months |
| `default_12m` | a customer's history at 2024-07-01 | 1 if the customer defaults within a year |
| `ltv_positive` | a customer's history at 2025-01-01 | 1 if the next six months are profitable |

The book's own fraud labels can be spotted from one event (every one is an online payment abroad), so they can't show what a history adds. The two fraud tasks instead **splice identity theft** into the unmodified book. For 20% of customers, the card transactions and app sessions of a matched donor (same country, similar spending) are copied in on one day. Each copied payment is ordinary *for somebody*, so a detector has to notice that it doesn't fit *this* customer's history. `fraud_user` asks it of the customer, `fraud_txn` of each transaction. Notebook 01 explains the splice and the forecast tasks.

As in Blueprint v1's notebook 05, each task splits customers first, so a model is always scored on customers it never saw.

In [ ]:
from tfm.runtime.evaluate import fraud_entity_split, fraud_task

# The per-customer fraud task comes straight from the SDK's evaluation; lab_tasks builds the other four.
tasks = {"fraud_user": fraud_task(data, fraud_entity_split(data, config.seed)),
         **lab_tasks(assets, data, ("fraud_txn", "churn_6m", "default_12m", "ltv_positive"))}
support = pd.DataFrame({name: task.summary() for name, task in tasks.items()}).T
support["eval positive share"] = (support["n_eval_positive"] / support["n_eval"]).round(3)
support

The fraud tasks' positive shares are set by the splice, far above real card fraud, and `default_12m` has the fewest positives, so its scores are the noisiest. Compare models with each other on the same task, not with Blueprint v1's TabFormer numbers.

## 3. Event representation: key-value tokens

The **key-value tokenizer** writes each field as a pair: a key token naming the field (`KEY_amount`) and a value token (`VAL_amount_b9`). An event is the list of its pairs, so events from different sources just have different keys. Values become tokens in one of four ways, fixed per key when the tokenizer is fitted on the train side:

* **categorical** fields with few values (`txn_type`, `country`, `mcc`): one token per value
* **numeric** fields (`amount`): 16 percentile buckets, like v1's amount bins
* **hashed** identifiers with huge cardinality (`merchant`, `device_id`, `counterparty`): `hash(key, value) mod 4096`, so a merchant never seen in training still gets a real token
* any other text: one token per byte

In [ ]:
sequence = max(data.test_sequences, key=lambda s: len({e.categorical.get("source") for e in s.events}))
record = tokenizer.encode(sequence)
n_events = len(record.event_offsets) - 1
encoded = sequence.events[-n_events:]
sources = [e.categorical.get("source") for e in encoded]
names = tokenizer.vocabulary.id_to_token
for j in [sources.index(s) for s in dict.fromkeys(sources)][:3]:
    start, end = record.event_offsets[j], record.event_offsets[j + 1]
    print(f"event {j}: {sources[j]} at {encoded[j].event_timestamp:%Y-%m-%d %H:%M}")
    for key, value in zip(record.key_ids[start:end], record.value_ids[start:end]):
        field = names[key].removeprefix("KEY_")
        print(f"    {names[key]:<18} {names[value]:<30} {tokenizer.vocabulary.field_kind.get(field, '')}")
print(f"\nvocabulary: {tokenizer.vocab_size:,} tokens; median tokens per event: {int(np.median(np.diff(record.event_offsets)))}")

**Time is not a token.** v1 spent three tokens per transaction on hour, day of week, and month. v2 embeds each event's calendar (hour, weekday, day of month) and adds it to the event's vector. The decoder then rotates attention by **elapsed time** with **TimeRoPE**: an event's position is `8·ln(1 + Δt/8)`, where `Δt` is the seconds to the sequence's last event. Two taps a second apart and two payments a month apart are no longer both "one step apart".

In [ ]:
for i in np.linspace(0, n_events - 1, 6).astype(int):
    print(f"event {i:>3}: {sources[i]:<10} {encoded[i].event_timestamp:%Y-%m-%d %H:%M}  "
          f"hour {record.events['hour'][i]:>2}, weekday {record.events['dow'][i]}, day {record.events['dom'][i]:>2}, TimeRoPE position {record.time_log[i]:6.2f}")

The last event sits at position 0 and earlier events at larger positions. The log keeps resolution for short gaps and compresses long ones. Notebook 02 plots the buckets, the hash collisions, and the log-time transform.

## 4. The network

```text
profile tokens (as of cutoff) -> profile encoder (bidirectional) -> z_profile
event j key/value tokens      -> event encoder (bidirectional)   -> z_event[j]
[z_profile, z_event[0..N-1]] + calendar, TimeRoPE -> causal decoder -> h[0..N]
h at the slot after the last event -> embedding
```

* The **event encoder** attends only inside one event, a handful of tokens, so the expensive token-level attention stays small.
* The **causal decoder** attends over one vector per event. Slot 0 is the profile and slot `j + 1` is event `j`; each slot sees only itself and earlier slots. That causal mask is what makes the embedding at a cutoff safe for forecasting, exactly as in Blueprint v1's notebook 04.
* The SDK composes it with `tfm.experimental.compose_blueprint_v2`, the v2 counterpart of the `TrxFM(...)` call in Blueprint v1's notebook 03.

In [ ]:
g = config.model.composition.granularity
print(f"hidden {g.decoder.hidden_size}, {g.decoder.num_heads} heads; depths: profile encoder {g.profile_depth}, "
      f"event encoder {g.event_depth}, decoder {g.decoder.depth}; up to {config.data.sequence_length} events per row")

params = {name: sum(p.numel() for p in module.parameters()) for name, module in model.named_children()}
params = pd.Series({k: v for k, v in params.items() if v}).sort_values()
fig, ax = plt.subplots(figsize=(9, 0.3 * len(params) + 1.2))
ax.barh(params.index, params.to_numpy())
for i, n in enumerate(params.to_numpy()):
    ax.annotate(f" {n / 1e6:.2f}M", (n, i), va="center", fontsize=8)
ax.set_xlabel("parameters")
ax.set_title(f"Blueprint v2 lab checkpoint: {params.sum() / 1e6:.1f}M parameters by module")
plt.tight_layout()
plt.show()

The token embedding table (`embed`) is the largest module, mostly hash buckets. The three transformer stacks share the rest. At 256 hidden units the network is far smaller than Blueprint v1's 25.9M-parameter Llama (v1 notebook 03), yet it reads the same 128 events, because each event costs one decoder position instead of 12 tokens.

## 5. What pretraining taught the model

**Objective.** `next_event_fields`: the decoder state that has seen the profile and events `0…j-1` predicts every field value of event `j`. This is v1's next-token idea, lifted from tokens to events. The auxiliary `token_event_key_mlm` is PRAGMA's masked objective: it hides tokens, whole events, and whole keys inside each event, and asks the event encoder to restore them. The losses are summed.

The checkpoint pretrained for 3,000 steps (about 42 minutes on one GB10) on every 128-event window of all 100,000 customers. Here we score the pretrained and the untrained network on 256 held-out validation customers.

In [ ]:
from tfm.adaptation.batching import minibatches, select_rows

held_out = select_rows(data.batches[1], list(range(min(256, data.batches[1].batch_size))))
losses = {}
for name, m in {"pretrained": model, "random init": random_model}.items():
    torch.manual_seed(config.seed)
    parts = []
    with torch.no_grad():
        for chunk in minibatches(held_out, 16):
            parts.append({k: float(v) for k, v in m(chunk.to(assets.device)).losses.items()})
    losses[name] = pd.DataFrame(parts).mean()
losses = pd.DataFrame(losses)

fig, ax = plt.subplots(figsize=(8, 3.2))
x = np.arange(len(losses))
ax.bar(x - 0.2, losses["random init"], 0.4, color="lightgray", label="random init")
ax.bar(x + 0.2, losses["pretrained"], 0.4, color="#76B900", label="pretrained")
ax.set_xticks(x, losses.index)
ax.set_yscale("log")
ax.set_ylabel("loss (log scale)")
ax.set_title("Held-out loss per objective")
ax.legend()
plt.tight_layout()
plt.show()
losses.round(3)

Pretraining cuts both losses by more than an order of magnitude: the untrained network is confidently wrong about every field, and the pretrained one is close to the data. Notebook 03b breaks the next-event loss down field by field against two floors (a key's overall frequency, and copying the customer's previous value), and pretrains from scratch for a few minutes so you can watch the loss fall.

## 6. Fine-tune and score on five tasks

**The readout** is the same idea as Blueprint v1's notebook 04: the **embedding** is the decoder state at the slot after the row's last event, 256 numbers that summarize the profile and the history up to the row's cutoff.

**The baseline** is Blueprint v1's raw arm (v1 notebook 05): XGBoost on the row's last event's fields and the customer's profile, with no history. One XGBoost setting is shared by every task (`LAB_XGBOOST_PARAMS`; early stopping picks the tree count).

**The TFM** adapts the pretrained network to each task with **LoRA** (low-rank adaptation). Every pretrained weight stays frozen; small rank-16 updates train beside the attention and feed-forward layers of all three stacks, about 5% of the parameters, and a linear head reads the embedding. Each task gets its own copy of the checkpoint. Training stops early once the loss on 20% of the train-side customers, held out, stops improving; the evaluation customers are never seen.

**The metric** is PR-AUC, which suits rare positives. `ltv_positive` is the exception: about 71% of customers are positive, so its PR-AUC starts near 0.9 for any model, and ROC-AUC shows the difference better.

In [ ]:
from tfm.adaptation.finetune import finetune_binary
from tfm.evaluation.tabular import evaluate_raw_embedding_arms
from tfm.runtime.evaluate import finetune_spec
from tfm.runtime.lab import LAB_XGBOOST_PARAMS

METRIC = {name: "roc_auc" if name == "ltv_positive" else "pr_auc" for name in tasks}
spec = finetune_spec(config)  # fraud_task.finetune in the checkpoint's config.yaml
baselines = config.evaluation.tabular_baselines
results = {}
started = time.perf_counter()
for name, task in tasks.items():
    raw, _ = evaluate_raw_embedding_arms(
        task.train_sequences, task.y_train, task.eval_sequences, task.y_eval,
        profiles=profiles, embedding_views={},
        xgboost_params=LAB_XGBOOST_PARAMS, seed=config.seed, top_k=baselines.top_k,
        coded_keys=tuple(config.features.categorical_keys), n_jobs=baselines.n_jobs,
        k=config.evaluation.precision_at_k,
    )
    lora, manifest = finetune_binary(
        model, task.train_batch, task.eval_batch, task.y_train, task.y_eval, spec,
        device=assets.device, seed=config.seed, k=config.evaluation.precision_at_k,
        precision=config.training.precision,
    )
    results[name] = {"metric": METRIC[name], "raw": raw["xgboost_raw"][METRIC[name]],
                     "LoRA": lora[METRIC[name]], "best step": manifest.get("best_step")}
print(f"LoRA rank {spec.lora.rank} on {', '.join(spec.lora.targets)}: "
      f"{manifest.get('n_trainable_params', 0):,} trainable parameters per task; "
      f"{len(tasks)} tasks in {time.perf_counter() - started:.0f} s")

scores = pd.DataFrame(results).T.astype({"raw": float, "LoRA": float})
scores["lift"] = (scores["LoRA"] / scores["raw"] - 1).map("{:+.0%}".format)
scores.round(3)

In [ ]:
LABELS = {"fraud_user": "fraud, per customer", "fraud_txn": "fraud, per txn", "churn_6m": "churn 6m",
          "default_12m": "default 12m", "ltv_positive": "LTV positive"}

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(scores))
ax.bar(x - 0.2, scores["raw"], 0.4, color="lightgray", label="XGBoost, raw features")
ax.bar(x + 0.2, scores["LoRA"], 0.4, color="#76B900", label="TFM, LoRA fine-tuned")
for j, name in enumerate(scores.index):
    ax.annotate(scores.loc[name, "lift"], (j + 0.2, scores.loc[name, "LoRA"]),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
    ax.hlines(tasks[name].y_eval.mean() if METRIC[name] == "pr_auc" else 0.5,
              j - 0.4, j + 0.4, color="black", ls=":", lw=1)
ax.set_xticks(x, [f"{LABELS[n]}\n{METRIC[n].replace('_', '-').upper()}" for n in scores.index])
ax.set_ylim(0, 1.05)
ax.set_title("Raw features against the fine-tuned TFM (dotted: a random ranking)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

How to read it, with Blueprint v1's caution: this is one 4,000-customer sample and one customer split, so a task with few evaluation positives moves from sample to sample. Read each task against its own dotted line. The recorded result, at three seeds on 10,000 customers, is [`blueprint_v2_final_sweep`](../../benchmarks/results/blueprint_v2_final_sweep.md): this checkpoint's setting (candidate D) lifted raw-feature XGBoost with LoRA on fraud, fraud per transaction, churn, and LTV at every seed; `default_12m` is reported there, not scored, for want of positives.

* **One pretrained model serves every task.** The same checkpoint, fine-tuned
  for a few hundred steps per task on about 5% of its weights, is scored
  against the raw-feature model on fraud, churn, default, and lifetime value.
  Pretraining saw no label of any task. Read the four headline tasks against
  the recorded run above, where this setting beat the raw model at every
  seed, and `default_12m` as below.
* **The lift is what the history adds.** The baseline sees only the latest
  event and the profile, by design; the TFM reads the last 128 events. The
  largest gains are where the evidence is spread over many events, such as a
  day of payments that don't fit the customer's past.
* **Small tasks are noisy.** `default_12m` rests on about 25 evaluation
  positives in this sample, and its lift changes sign from one sample, split,
  or baseline to the next (it did between SDK 2.0.0a1, whose raw baseline
  coded fewer categorical values, and this release). Treat it as no result.

Compare this with Blueprint v1's notebook 05. There, the history embedding barely helped v1's fraud model. Here, a model that reads one vector per event, fine-tuned briefly per task, improves on a classic model on held-out customers on the four tasks with enough positives to tell. The datasets differ, so this isn't a head-to-head result; `benchmarks/ablations/blueprint_v2_lab_v1_flat.yaml` runs v1's flat decoder on these same rows for that comparison.

## 7. Go further

Each section above has a notebook in this series ([README](README.md)). They run on the same book and checkpoint, about 17 minutes in total on one GPU:

| Notebook | Building blocks | Goes further with |
|---|---|---|
| [01 · Data and labels](01_data_and_labels.ipynb) | data | the book's sources and fields, the three cutoffs and label windows, the identity-theft splice step by step, all five tasks |
| [02 · Event representation](02_event_representation.ipynb) | event representation | percentile buckets, hash collisions and why hashing mattered, the log-time transform, the event-boundary rule |
| [03b · Architecture and objective](03b_blueprint_v2_decoder.ipynb) | architecture, objective | attention masks, per-field loss against its floors, compositions the SDK refuses, **a short pretraining run**, how the checkpoint was configured |
| [04 · Readout and adaptation](04_readout_and_adaptation.ipynb) | readout, adaptation | the embedding space, and **LoRA fine-tuning** on one task step by step: early stopping, what the adapters train, rank and head |

To reproduce the checkpoint itself (about 4 minutes of token caching and 42 of pretraining on one GB10), train from the config it was trained with, final-sweep candidate D ([record](../../benchmarks/results/blueprint_v2_final_sweep.md)); [05 · Scaling up](05_scaling_up.ipynb) has the commands to package your own:

```bash
export TFM_PRAGMATIQ_ROOT=/path/to/synth_100k_gen2_m31_seed0
uv run tfm cache-tokens --recipe blueprint_v2 --workers 6
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
uv run tfm report --config benchmarks/ablations/blueprint_v2_final_d.yaml \
  --device cuda --run-dir outputs/ablations/blueprint_v2_final_d
```

## Summary
In this notebook we:
1. **Compared** Blueprint v1 and v2 building block by building block: v2 compresses each event, and the profile, into one vector before a causal decoder reads the history
2. **Met** a multi-source synthetic book with customer profiles, and five tasks on the same customers: two spliced identity-theft tasks, churn, default, and lifetime value
3. **Tokenized** an event as key-value pairs with hashed identifiers, with time entering as a rotary position rather than a token
4. **Opened** the network: profile and event encoders under a causal decoder, 129 positions for 128 events
5. **Checked** that pretraining cut every held-out loss against the same network untrained
6. **Fine-tuned** the pretrained model with LoRA on each task and scored it against XGBoost on raw features: the TFM lifts the four headline tasks, at every seed in the recorded run; `default_12m` has too few positives to score

## What these numbers do not show
- Anything about real customers. The book is synthetic and the fraud is spliced in; this is a toy problem, built to show how a TFM pipeline is set up, trained, and tested on several downstream tasks.
- A range. One pretraining seed, one architecture, one sample, one customer split.
- A head-to-head of v1 and v2. The datasets differ; `benchmarks/ablations/blueprint_v2_lab_v1_flat.yaml` runs v1 on these rows.
- A production or fraud-detection claim of any kind.

## Takeaways
- The sequence architecture decides what a history costs: one position per token (v1) or one per event (v2).
- A key-value tokenizer reads events of any shape, so one model can read every source a bank has.
- One self-supervised pretraining run serves many tasks: a short LoRA fine-tune per task, about 5% of the parameters, adapts the same checkpoint to each.
- Keep the controls from Blueprint v1's notebooks 04 and 05: customer-disjoint splits, one fixed classifier setting for the baseline, early stopping on held-out train-side customers, and an untrained network to check that pretraining did something.

## Where this lives in the SDK
This notebook explains; the recipe runs. Every cell calls `src/tfm`, so the same code runs in [`recipes/blueprint_v2`](../../recipes/blueprint_v2).

Assets and tasks: `tfm.runtime.lab` (`fetch_lab_assets`, `prepare_lab_data`, `load_lab_model`, `lab_tasks`, `LAB_XGBOOST_PARAMS`) and `tfm.runtime.evaluate.fraud_task`. Tokenizer: `tfm.features.tokenizers.key_value_time`. Model: `tfm.layers.profile_event_decoder`, composed by `tfm.experimental.compose_blueprint_v2`. Objective: `tfm.objectives.next_event_fields`. Fine-tuning: `tfm.adaptation.finetune.finetune_binary`, with the settings `tfm.runtime.evaluate.finetune_spec` reads from `fraud_task.finetune`. Baseline: `tfm.evaluation.tabular.evaluate_raw_embedding_arms`.

Next: [01 · Data and labels](01_data_and_labels.ipynb).